# 3.1 Merge individual and panel datasets

This notebook does the following:
    - Calculates per equipment type and total electricity consumption for BL and EL
    - Merges the individual data with the electricity data
    - Cleans up and saves the dataset

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
import importlib
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config
import os
import statsmodels.formula.api as smf
import pyfixest as pf
from make_regression_table import make_regression_table

In [2]:
# Load data
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv", 
    keep_default_na=False, # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Construct post variable
df["post"] = (df["survey"] == "EL").astype(int)

In [4]:
# Keep only labgroups that have both pre and post observations
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
labgroups_to_keep = labgroup_counts[labgroup_counts == 2].index
df = df[df["labgroupid"].isin(labgroups_to_keep)].copy()

## (1) Simple diff in diff regression


In [5]:
fit_levels = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)

fit_levels.summary()

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


###

Estimation:  OLS
Dep. var.: annual_electricity_total, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |     2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|---------:|--------:|
| treated:post  |    -19.563 |      111.782 |    -0.175 |      0.861 | -241.136 | 202.009 |
---
RMSE: 288.413 R2: 1.0 R2 Within: 0.0 


In [6]:
# How much does electricity vary within individuals?
df.groupby('labgroupid')['annual_electricity_total'].std().describe()

# Are there any individuals with zero variation across periods?
no_var = df.groupby('labgroupid')['annual_electricity_total'].std() == 0
print(f"\nIndividuals with identical BL and EL outcome: {no_var.sum()}")


Individuals with identical BL and EL outcome: 86


In [7]:
df['log_electricity'] = np.log1p(df['annual_electricity_total']) 

fit_log = pf.feols(
    "log_electricity ~ treated:post | labgroupid + post",
    data=df,
    vcov={"CRV1": "labgroupid"}
)
fit_log.summary()

###

Estimation:  OLS
Dep. var.: log_electricity, Fixed effects: labgroupid + post
sample: None = all
Inference:  CRV1
Observations:  218

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treated:post  |     -0.053 |        0.051 |    -1.038 |      0.302 | -0.153 |   0.048 |
---
RMSE: 0.127 R2: 0.994 R2 Within: 0.011 


## (3) Robustness checks

### (3a) Alternative fixed effects specifications

In [8]:
# Robustness: faculty-, institute-, and enumerator-time FEs
# X^post creates X-specific time dummies (X × post interactions)

fit_levels_faculty = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + faculty^post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_faculty = pf.feols(
    "log_electricity ~ treated:post | labgroupid + faculty^post",
    data=df, vcov={"CRV1": "labgroupid"}
)

fit_levels_institute = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + institute_id^post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_institute = pf.feols(
    "log_electricity ~ treated:post | labgroupid + institute_id^post",
    data=df, vcov={"CRV1": "labgroupid"}
)

fit_levels_enum = pf.feols(
    "annual_electricity_total ~ treated:post | labgroupid + enum_id^post",
    data=df, vcov={"CRV1": "labgroupid"}
)
fit_log_enum = pf.feols(
    "log_electricity ~ treated:post | labgroupid + enum_id^post",
    data=df, vcov={"CRV1": "labgroupid"}
)

/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 22 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 22 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 10 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/drutna/miniconda3/envs/labrct/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:149: UserWarning: 10 singleton fixed effect(s) dropped from the model.
  warnings.warn(


In [9]:
# Export to nice table
table = make_regression_table(
    fit_list = [
        fit_levels, fit_levels_faculty, 
        fit_levels_institute, fit_levels_enum,
        fit_log, fit_log_faculty,
        fit_log_institute, fit_log_enum,
    ],
    model_names   = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)", "(8)"],
    keep_vars     = ["treated:post"],
    var_labels    = {"treated:post": "Treated $\\times$ Post"},
    fe_rows       = {
        "Lab group FE":                  [True] * 8,
        "Time FE":                       [True, False, False, False, True, False, False, False],
        "Faculty $\\times$ Time FE":     [False, True, False, False, False, True, False, False],
        "Institute $\\times$ Time FE":   [False, False, True, False, False, False, True, False],
        "Enumerator $\\times$ Time FE":  [False, False, False, True, False, False, False,  True],
    },
    col_groups    = {"Levels": [0,1,2,3], "Log": [4,5,6,7]},
    col_subgroups = {"Baseline": [0,4], "Robustness": [1,2,3,5,6,7]},
    baseline_mean  = "auto",
    outcome_levels = "annual_electricity_total",
    df_levels      = df,
    decimals       = [1, 1, 1, 1, 3, 3, 3, 3],
    r2_type        = None,
    col1_width     = "4cm",
    coln_width     = "1.5cm",
)
table_path = config.OUTPUT / "5_Regression_Tables" / "simple_diff_in_diff.tex"
_ = table_path.write_text(table)